# Goal

  This notebook builds the final modeling workflow for the IEEE-CIS Fraud Detection project.

  It uses the feature decisions from previous notebooks:
  - notebook `02`: selected top Vesta `V*` features
  - notebook `03`: selected top identity `id_*` features and useful missingness indicators

  This notebook focuses on:
  - building the final train/validation dataset
  - applying preprocessing for baseline and boosted models
  - comparing Logistic Regression, LightGBM, and XGBoost
  - selecting an operating classification threshold
  - defining what will be exported into the reproducible pipeline

  This is a modeling notebook, not an EDA notebook.

> Run this notebook in Google Colab. Kaggle credentials are loaded from Colab secrets.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import plotly.express as px

try:
    from google.colab import userdata
    key = userdata.get("KAGGLE_API_TOKEN")
    if key:
        os.environ["KAGGLE_API_TOKEN"] = key.strip()
except ImportError:
    pass

In [2]:
# Install the Kaggle CLI only when this Colab runtime needs it.
import importlib.metadata
import subprocess
import sys

try:
    kaggle_version = importlib.metadata.version("kaggle")
except importlib.metadata.PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    kaggle_version = importlib.metadata.version("kaggle")
print("Kaggle CLI:", kaggle_version)

Kaggle CLI: 2.0.2


In [3]:
# Pin the XGBoost version used for the Colab feature-selection artifacts.
import importlib.metadata
import subprocess
import sys
try:
    installed_xgb = importlib.metadata.version("xgboost")
except importlib.metadata.PackageNotFoundError:
    installed_xgb = None
if installed_xgb != "3.4.1":
    if "xgboost" in sys.modules:
        raise RuntimeError("Restart the Colab runtime before installing XGBoost 3.4.1")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost==3.4.1"])
import xgboost as xgb
assert xgb.__version__ == "3.4.1"
print("Feature-selection XGBoost:", xgb.__version__)

Feature-selection XGBoost: 3.4.1


In [4]:
SEED = 42

# Helper functions

In [5]:
import warnings
from contextlib import nullcontext

def reduce_memory(df: pd.DataFrame, verbose: bool = True, ignore_warnings: bool = False, inplace: bool = True) -> pd.DataFrame:
    """
    Downcast numeric columns to reduce DataFrame memory usage.
    """
    if not inplace:
        df = df.copy()

    start_mem = df.memory_usage(deep=True).sum() / 1e6

    warning_context = warnings.catch_warnings() if ignore_warnings else nullcontext()

    with warning_context:
        if ignore_warnings:
            warnings.filterwarnings(
                "ignore",
                category=RuntimeWarning,
                message="overflow encountered in cast",
            )

        for col in df.columns:
            series = df[col]

            if pd.api.types.is_integer_dtype(series):
                c_min = series.min()
                c_max = series.max()

                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = series.astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = series.astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = series.astype(np.int32)
                else:
                    df[col] = series.astype(np.int64)

            elif pd.api.types.is_float_dtype(series):
                c_min = series.min()
                c_max = series.max()

                if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                    df[col] = series.astype(np.float32)
                else:
                    df[col] = series.astype(np.float64)

    end_mem = df.memory_usage(deep=True).sum() / 1e6

    if verbose:
        reduction = 100 * (start_mem - end_mem) / start_mem if start_mem else 0
        print(f"Memory: {start_mem:.2f} MB → {end_mem:.2f} MB ({reduction:.1f}% reduction)")

    return df

# Data load

In [6]:
import subprocess

archive = Path("/content/ieee-fraud-detection.zip")
required_csvs = [Path("/content/train_transaction.csv"), Path("/content/train_identity.csv")]
if not all(path.exists() for path in required_csvs):
    if not archive.exists():
        subprocess.run(
            ["kaggle", "competitions", "download", "-c", "ieee-fraud-detection", "-p", "/content"],
            check=True,
        )
    subprocess.run(["unzip", "-qo", str(archive), "-d", "/content"], check=True)
if not all(path.exists() for path in required_csvs):
    raise FileNotFoundError("Kaggle training CSVs are missing from /content")
print("Training CSVs ready in /content")


Training CSVs ready in /content


In [7]:
import json
import hashlib
from google.colab import drive

drive.mount("/content/drive")
BASE_PATH = Path("/content/")
TRAIN_TRANSACTION = BASE_PATH / "train_transaction.csv"
TRAIN_IDENTITY = BASE_PATH / "train_identity.csv"
ARTIFACTS_PATH = Path("/content/drive/My Drive/ML/Fraud Detection/Data/Artifacts")
with open(ARTIFACTS_PATH / "selected_v_features.json") as f:
    selected_v_features = json.load(f)
with open(ARTIFACTS_PATH / "selected_id_features.json") as f:
    selected_id_features = json.load(f)
with open(ARTIFACTS_PATH / "v_feature_selection_metadata.json") as f:
    v_selection_metadata = json.load(f)
with open(ARTIFACTS_PATH / "id_feature_selection_metadata.json") as f:
    id_selection_metadata = json.load(f)

# Reject a stale or changed Drive list before loading the wide CSVs.
for name, features, metadata in [
    ("V", selected_v_features, v_selection_metadata),
    ("ID", selected_id_features, id_selection_metadata),
]:
    actual_sha256 = hashlib.sha256(json.dumps(features, separators=(",", ":")).encode()).hexdigest()
    if (metadata.get("feature_sha256") != actual_sha256
            or metadata.get("xgboost_version") != "3.4.1"):
        raise ValueError(f"Stale {name} features in Drive. Rerun notebooks 2 and 3 with XGBoost 3.4.1.")

# Confirm both selectors and this model notebook use the same raw files.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

for source_name, path in [("transaction", TRAIN_TRANSACTION), ("identity", TRAIN_IDENTITY)]:
    actual_sha256 = sha256_file(path)
    key = f"{source_name}_csv_sha256"
    if (v_selection_metadata.get(key) != actual_sha256
            or id_selection_metadata.get(key) != actual_sha256):
        raise ValueError(f"{source_name} CSV differs from the files used in notebooks 2 and 3")

# The selected ID list can include derived missing indicators. Read their source columns.
selected_id_source = {name.removesuffix("_is_missing") for name in selected_id_features}


Mounted at /content/drive


In [8]:
import gc

# Parse only columns used by the models. This avoids holding all 339 V columns
# and all identity columns while subsequently selecting a smaller frame.
train_transaction = reduce_memory(pd.read_csv(
    TRAIN_TRANSACTION,
    usecols=lambda col: not col.startswith("V") or col in selected_v_features,
), True)
train_identity = reduce_memory(pd.read_csv(
    TRAIN_IDENTITY,
    usecols=lambda col: not col.startswith("id_") or col in selected_id_source,
), True)
train_df_merged = pd.merge(train_transaction, train_identity, on="TransactionID", how="left")
del train_transaction, train_identity
gc.collect()
print("Loaded", len(train_df_merged), "transactions with selected features")


Memory: 1642.56 MB → 1001.83 MB (39.0% reduction)
Memory: 124.18 MB → 114.95 MB (7.4% reduction)
Loaded 590540 transactions with selected features


# Feature reduction

Rerun notebooks 2 and 3 before this notebook. Their supervised feature selectors fit only on the earlier 80% of transactions and save lists plus selection metadata to the Artifacts folder. This notebook verifies the training-period boundary before loading either list. The final 20% time holdout is reserved for model evaluation.

In [9]:
# Fail fast if the lists were made for a different period or split.
time_order = train_df_merged["TransactionDT"].sort_values(kind="stable").index
split_idx = int(len(time_order) * 0.8)
expected_train_idx, expected_holdout_idx = time_order[:split_idx], time_order[split_idx:]
expected_time_max = float(train_df_merged.loc[expected_train_idx, "TransactionDT"].max())
expected_holdout_min = float(train_df_merged.loc[expected_holdout_idx, "TransactionDT"].min())
for name, metadata, features in [
    ("V", v_selection_metadata, selected_v_features),
    ("ID", id_selection_metadata, selected_id_features),
]:
    if (metadata.get("xgboost_version") != "3.4.1"
            or metadata["n_selection_rows"] != len(expected_train_idx)
            or metadata["selection_time_max"] != expected_time_max
            or metadata["holdout_time_min"] != expected_holdout_min
            or metadata["n_selected_features"] != len(features)
            or metadata["gain_fraction"] != 0.95):
        raise ValueError(f"Stale {name} feature list. Rerun notebooks 2 and 3 in order.")
print("Validated training-period feature lists:", len(selected_v_features), "V and", len(selected_id_features), "ID")


Validated training-period feature lists: 229 V and 28 ID


In [10]:
assert set(col for col in train_df_merged if col.startswith("V")) <= set(selected_v_features)


In [11]:
# Materialize only selected missing indicators, without copying the whole table.
for col in selected_id_features:
    if col.endswith("_is_missing"):
        source_col = col.removesuffix("_is_missing")
        train_df_merged[col] = train_df_merged[source_col].isna().astype("int8")


In [12]:
# An indicator may be selected without its raw ID source column.
unselected_id_sources = [
    col for col in train_df_merged
    if col.startswith("id_") and col not in selected_id_features
]
train_df_merged.drop(columns=unselected_id_sources, inplace=True)
assert set(col for col in train_df_merged if col.startswith("id_")) <= set(selected_id_features)


In [13]:
TARGET = "isFraud"
DROP_COLS = ["TransactionID"]
TIME_COL = "TransactionDT"

#Train/validation split

In [14]:
X = train_df_merged.drop(columns=[TARGET])
y = train_df_merged[TARGET]
split_idx = int(len(X) * 0.8)
time_order = X[TIME_COL].sort_values(kind="stable").index
train_idx, val_idx = time_order[:split_idx], time_order[split_idx:]
X_train, X_val = X.loc[train_idx], X.loc[val_idx]
y_train, y_val = y.loc[train_idx], y.loc[val_idx]
inner_order = X_train[TIME_COL].sort_values(kind="stable").index
print("Train period:", X_train[TIME_COL].min(), "-", X_train[TIME_COL].max())
print("Validation period:", X_val[TIME_COL].min(), "-", X_val[TIME_COL].max())
del X, y, train_df_merged
gc.collect()


Train period: 86400 - 12192842
Validation period: 12192900 - 15811131


0

# Feature grouping validation

In [15]:
id_like_numeric = [
  "card1", "card2", "card3", "card5",
  "addr1", "addr2"
]

known_categorical = [
  "ProductCD",
  "card4", "card6",
  "P_emaildomain", "R_emaildomain",
  "DeviceType", "DeviceInfo"
]


In [16]:
from pandas.api.types import (
    is_object_dtype,
    is_bool_dtype,
    is_numeric_dtype,
    is_string_dtype,
)
from pandas import CategoricalDtype

def is_categorical_like(series):
    return (
        is_object_dtype(series)
        or isinstance(series.dtype, CategoricalDtype)
        or is_bool_dtype(series)
        or is_string_dtype(series)
    )

def get_feature_groups(df, selected_id_features):
  numeric_cols = []
  categorical_cols = []

  for col in df.columns:
    if col in ["TransactionID", "isFraud"]:
      continue

    if col in id_like_numeric or col in known_categorical:
      categorical_cols.append(col)
      continue

    if col in selected_id_features:
      if is_categorical_like(df[col]):
          categorical_cols.append(col)
      else:
          numeric_cols.append(col)
      continue

    if is_categorical_like(df[col]):
      categorical_cols.append(col)
    elif is_numeric_dtype(df[col]):
      numeric_cols.append(col)

  numeric_cols = [c for c in numeric_cols if c != "TransactionDT"]
  return numeric_cols, categorical_cols

| Feature type                     | Baseline model                             | Tree model                                       |
| -------------------------------- | ------------------------------------------ | ------------------------------------------------ |
| Low-cardinality categorical      | One-hot encoding                           | One-hot or native categorical                    |
| High-cardinality categorical     | Rare grouping + one-hot/frequency encoding | Rare grouping + ordinal/category/native handling |
| `DeviceInfo`                     | Rare grouping first                        | Rare grouping first                              |
| `P_emaildomain`, `R_emaildomain` | Group rare domains                         | Group rare domains                               |
| `card/addr` encoded IDs          | Treat as categorical                       | Treat as categorical                             |


In [17]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression


class RareCategoryGrouper(BaseEstimator, TransformerMixin):
  def __init__(self, min_frequency=0.01, missing_value="Missing"):
    self.min_frequency = min_frequency
    self.missing_value = missing_value
    self.frequent_categories_ = {}

  def fit(self, X, y=None):
    X = pd.DataFrame(X)

    for col in X.columns:
        s = X[col].fillna(self.missing_value).astype(str)
        freq = s.value_counts(normalize=True)
        self.frequent_categories_[col] = set(freq[freq >= self.min_frequency].index)

    return self

  def transform(self, X):
    X = pd.DataFrame(X)

    for col in X.columns:
        s = X[col].fillna(self.missing_value).astype(str)
        keep = self.frequent_categories_[col]
        X[col] = s.where(s.isin(keep), "Rare")

    return X

In [18]:
def add_features(df, use_log: bool = True):
  # drop already returns a new frame; avoid copying it a second time.
  df = df.drop(columns=DROP_COLS, errors="ignore")
  if "TransactionDT" in df.columns:
      df["Transaction_day"] = df["TransactionDT"] // (60 * 60 * 24)
      df["Transaction_hour"] = (df["TransactionDT"] // (60 * 60)) % 24

  if "TransactionAmt" in df.columns and use_log:
      df["TransactionAmt_log1p"] = np.log1p(df["TransactionAmt"])

  return df


In [19]:
X_train_fe = add_features(X_train, use_log=True)
X_val_fe = add_features(X_val, use_log=True)
del X_train, X_val
gc.collect()


0

In [20]:
numeric_cols, categorical_cols = get_feature_groups(X_train_fe, selected_id_features)

In [21]:
missing_selected_id = sorted(set(selected_id_features) - set(X_train_fe.columns))
if missing_selected_id:
    raise ValueError(f"Selected ID features missing from modeling data: {missing_selected_id}")
missing_assignment = [
    col for col in selected_id_features
    if col not in numeric_cols and col not in categorical_cols
]
if missing_assignment:
    raise ValueError(f"Selected ID features lack preprocessing: {missing_assignment}")
print("Preprocessing columns:", len(numeric_cols), "numeric,", len(categorical_cols), "categorical")


Preprocessing columns: 280 numeric, 34 categorical


# Baseline model

In [22]:
numeric_pipeline = Pipeline(steps=[
  ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
  ("scaler", StandardScaler())
])

In [23]:
categorical_pipeline = Pipeline(steps=[
  ("rare", RareCategoryGrouper(min_frequency=0.005)),
  ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
  ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

In [24]:
preprocessor_baseline = ColumnTransformer(
  transformers=[
      ("num", numeric_pipeline, numeric_cols),
      ("cat", categorical_pipeline, categorical_cols)
  ],
  remainder="drop"
)

In [25]:
baseline_model = Pipeline(steps=[
  ("preprocess", preprocessor_baseline),
  ("model", LogisticRegression(
      max_iter=1000,
      class_weight="balanced",
      n_jobs=-1,
      random_state=SEED
  ))
])

In [26]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

baseline_model.fit(X_train_fe, y_train)

y_val_pred = baseline_model.predict(X_val_fe)
y_val_proba = baseline_model.predict_proba(X_val_fe)[:, 1]

In [27]:
baseline_metrics = {
  "model"     : "LogisticRegression",
  "accuracy"  : accuracy_score(y_val, y_val_pred),
  "precision" : precision_score(y_val, y_val_pred),
  "recall"    : recall_score(y_val, y_val_pred),
  "f1"        : f1_score(y_val, y_val_pred),
  "roc_auc"   : roc_auc_score(y_val, y_val_proba),
  "pr_auc"    : average_precision_score(y_val, y_val_proba),
}

baseline_metrics

{'model': 'LogisticRegression',
 'accuracy': 0.769270498188099,
 'precision': 0.10581075107952807,
 'recall': 0.765748031496063,
 'f1': 0.1859297983569828,
 'roc_auc': np.float64(0.8446265071714276),
 'pr_auc': np.float64(0.22423044790170896)}

In [28]:
print("ROC-AUC:", roc_auc_score(y_val, y_val_proba))
print("PR-AUC:", average_precision_score(y_val, y_val_proba))
print()
print(classification_report(y_val, y_val_pred))
print()
print(confusion_matrix(y_val, y_val_pred))

ROC-AUC: 0.8446265071714276
PR-AUC: 0.22423044790170896

              precision    recall  f1-score   support

           0       0.99      0.77      0.87    114044
           1       0.11      0.77      0.19      4064

    accuracy                           0.77    118108
   macro avg       0.55      0.77      0.53    118108
weighted avg       0.96      0.77      0.84    118108


[[87745 26299]
 [  952  3112]]


# Light Gradient Boosting

In [29]:
from lightgbm import LGBMClassifier
from sklearn.preprocessing import OrdinalEncoder

# Numeric pipeline (same as baseline)
numeric_pipeline_tree = Pipeline(steps=[
  ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
  # ("scaler", StandardScaler()) - No Need for scaling in tree models
])

# Categorical pipeline for tree models (using OrdinalEncoder)
categorical_pipeline_tree = Pipeline(steps=[
  ("rare", RareCategoryGrouper(min_frequency=0.005)),
  ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
  ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

# Preprocessor for the tree model
preprocessor_tree = ColumnTransformer(
  transformers=[
    ("num", numeric_pipeline_tree, numeric_cols),
    ("cat", categorical_pipeline_tree, categorical_cols)
  ],
  remainder="drop"
)

# Tree model pipeline using LightGBM
tree_model = Pipeline(steps=[
    ("preprocess", preprocessor_tree),
    ("model", LGBMClassifier(
        objective='binary',
        metric='average_precision',
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        random_state=SEED,
        n_jobs=-1,
        class_weight='balanced'
    ))
])

print("Training LightGBM model...")
tree_model.fit(X_train_fe, y_train)

y_val_pred_tree = tree_model.predict(X_val_fe)
y_val_proba_tree = tree_model.predict_proba(X_val_fe)[:, 1]

print("LightGBM Model Evaluation:")
print("ROC-AUC:", roc_auc_score(y_val, y_val_proba_tree))
print("PR-AUC:", average_precision_score(y_val, y_val_proba_tree))
print()
print(classification_report(y_val, y_val_pred_tree))
print()
print(confusion_matrix(y_val, y_val_pred_tree))

Training LightGBM model...
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 3.660324 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 27673
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 552
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM Model Evaluation:
ROC-AUC: 0.9047119336900497
PR-AUC: 0.5358025508524761

              precision    recall  f1-score   support

           0       0.99      0.95      0.97    114044
           1       0.32      0.65      0.43      4064

    accuracy                           0.94    118108
   macro avg       0.65      0.80      0.70    118108
weighted avg       0.96      0.94      0.95    118108


[[108449   5595]
 [  1417   2647]]


In [30]:
tree_metrics = {
  "model"     : "LightGBM",
  "accuracy"  : accuracy_score(y_val, y_val_pred_tree),
  "precision" : precision_score(y_val, y_val_pred_tree),
  "recall"    : recall_score(y_val, y_val_pred_tree),
  "f1"        : f1_score(y_val, y_val_pred_tree),
  "roc_auc"   : roc_auc_score(y_val, y_val_proba_tree),
  "pr_auc"    : average_precision_score(y_val, y_val_proba_tree),
}

tree_metrics

{'model': 'LightGBM',
 'accuracy': 0.9406306092728689,
 'precision': 0.3211599126425625,
 'recall': 0.6513287401574803,
 'f1': 0.43019665203965546,
 'roc_auc': np.float64(0.9047119336900497),
 'pr_auc': np.float64(0.5358025508524761)}

# XGBoost

In [31]:
from xgboost import XGBClassifier

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
scale_pos_weight_value = negative_count / positive_count

numeric_pipeline_xgb = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

categorical_pipeline_xgb = Pipeline(steps=[
    ("rare", RareCategoryGrouper(min_frequency=0.005)),
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

preprocessor_xgb = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline_xgb, numeric_cols),
        ("cat", categorical_pipeline_xgb, categorical_cols)
    ],
    remainder="drop"
)

xgb_model = Pipeline(steps=[
    ("preprocess", preprocessor_xgb),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        n_estimators=700,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=1,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        tree_method="hist",
        scale_pos_weight=scale_pos_weight_value
    ))
])

print("Training XGBoost model...")
xgb_model.fit(X_train_fe, y_train)

y_val_pred_xgb = xgb_model.predict(X_val_fe)
y_val_proba_xgb = xgb_model.predict_proba(X_val_fe)[:, 1]

print("XGBoost Model Evaluation:")
print("ROC-AUC:", roc_auc_score(y_val, y_val_proba_xgb))
print("PR-AUC:", average_precision_score(y_val, y_val_proba_xgb))
print()
print(classification_report(y_val, y_val_pred_xgb))
print()
print(confusion_matrix(y_val, y_val_pred_xgb))

Training XGBoost model...
XGBoost Model Evaluation:
ROC-AUC: 0.909574347832526
PR-AUC: 0.5343289016923092

              precision    recall  f1-score   support

           0       0.99      0.93      0.96    114044
           1       0.26      0.71      0.38      4064

    accuracy                           0.92    118108
   macro avg       0.63      0.82      0.67    118108
weighted avg       0.96      0.92      0.94    118108


[[106052   7992]
 [  1198   2866]]


In [32]:
xgb_metrics = {
  "model"     : "XGBoost",
  "accuracy"  : accuracy_score(y_val, y_val_pred_xgb),
  "precision" : precision_score(y_val, y_val_pred_xgb),
  "recall"    : recall_score(y_val, y_val_pred_xgb),
  "f1"        : f1_score(y_val, y_val_pred_xgb),
  "roc_auc"   : roc_auc_score(y_val, y_val_proba_xgb),
  "pr_auc"    : average_precision_score(y_val, y_val_proba_xgb),
}

xgb_metrics

{'model': 'XGBoost',
 'accuracy': 0.9221898601280184,
 'precision': 0.26395284582796097,
 'recall': 0.7052165354330708,
 'f1': 0.3841308135638654,
 'roc_auc': np.float64(0.909574347832526),
 'pr_auc': np.float64(0.5343289016923092)}

# Models Comparison

In [33]:
comparison_df = pd.DataFrame([baseline_metrics, tree_metrics, xgb_metrics])
comparison_df = comparison_df.sort_values("pr_auc", ascending=False).reset_index(drop=True)
comparison_df.to_csv(os.path.join(ARTIFACTS_PATH, "notebook_04_model_comparison.csv"), index=False)
comparison_df

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,LightGBM,0.940631,0.321160,0.651329,0.430197,0.904712,0.535803
1,XGBoost,0.922190,0.263953,0.705217,0.384131,0.909574,0.534329
2,LogisticRegression,0.769270,0.105811,0.765748,0.185930,0.844627,0.224230


PR-AUC is used as the main model-selection metric because the fraud label is highly imbalanced. ROC-AUC is still reported, but it can look strong even when performance on the minority fraud class is weaker.

F1 is used later only for threshold selection, not for choosing the model family.

# Threshold selection rule

Fit a separate copy of each tree model on the first 64% of all transactions and choose its threshold on the next 16%. Both slices lie within the earlier 80% training period. The final 20% time holdout is not used to choose a threshold. Review high-recall alternatives as an operational tradeoff.

The exported feature lists were ranked on the full earlier 80%, including this inner validation slice. Inner threshold scores can therefore be optimistic. A nested feature-selection fit would be required for an independent inner score.

# Threshold tuning for Light Gradient Boosting

In [34]:
from sklearn.base import clone

# Inner time split within the earlier 80% training period.
inner_cutoff = int(len(inner_order) * 0.8)
inner_train_idx, inner_val_idx = inner_order[:inner_cutoff], inner_order[inner_cutoff:]
X_inner_train_fe = X_train_fe.loc[inner_train_idx]
X_inner_val_fe = X_train_fe.loc[inner_val_idx]
y_inner_train, y_inner_val = y_train.loc[inner_train_idx], y_train.loc[inner_val_idx]

lightgbm_inner = clone(tree_model)
lightgbm_inner.fit(X_inner_train_fe, y_inner_train)
inner_lightgbm_scores = lightgbm_inner.predict_proba(X_inner_val_fe)[:, 1]
thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)
results = []
for threshold in thresholds:
    y_pred_threshold = (inner_lightgbm_scores >= threshold).astype(int)
    matrix = confusion_matrix(y_inner_val, y_pred_threshold)
    results.append({
        "threshold": threshold,
        "precision": precision_score(y_inner_val, y_pred_threshold, zero_division=0),
        "recall": recall_score(y_inner_val, y_pred_threshold, zero_division=0),
        "f1": f1_score(y_inner_val, y_pred_threshold, zero_division=0),
        "false_positives": int(matrix[0, 1]),
        "false_negatives": int(matrix[1, 0]),
    })
threshold_results = pd.DataFrame(results).sort_values("f1", ascending=False)
threshold_results.to_csv(os.path.join(ARTIFACTS_PATH, "notebook_04_lightgbm_inner_thresholds.csv"), index=False)
threshold_results.head(10)


[LightGBM] [Info] Number of positive: 12895, number of negative: 365050
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.985626 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 27615
[LightGBM] [Info] Number of data points in the train set: 377945, number of used features: 527
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,threshold,precision,recall,f1,false_positives,false_negatives
74,0.79,0.622519,0.491091,0.549049,1103,1885
73,0.78,0.607026,0.499190,0.547852,1197,1855
79,0.84,0.703264,0.447894,0.547254,700,2045
72,0.77,0.594304,0.507019,0.547203,1282,1826
78,0.83,0.684959,0.454914,0.546723,775,2019
76,0.81,0.650634,0.471112,0.546508,937,1959
71,0.76,0.581857,0.514309,0.546002,1369,1799
77,0.82,0.667317,0.461393,0.545571,852,1995
75,0.80,0.633989,0.478402,0.545315,1023,1932
80,0.85,0.720053,0.438175,0.544814,631,2081


In [35]:
best_threshold_row = threshold_results.iloc[0]
best_threshold_row

,74
threshold,0.790000
precision,0.622519
recall,0.491091
f1,0.549049
false_positives,1103.000000
false_negatives,1885.000000


In [36]:
high_recall_options = threshold_results[threshold_results["recall"] >= 0.85].sort_values("precision", ascending=False)
high_recall_options.head(10)

,threshold,precision,recall,f1,false_positives,false_negatives
8,0.13,0.123338,0.856641,0.215630,22553,531
7,0.12,0.118156,0.862311,0.207834,23838,510
6,0.11,0.112794,0.868790,0.199665,25312,486
5,0.10,0.108153,0.879590,0.192621,26866,446
4,0.09,0.103134,0.889309,0.184833,28645,410
3,0.08,0.097968,0.899298,0.176687,30670,373
2,0.07,0.092294,0.909557,0.167583,33134,335
1,0.06,0.086747,0.921706,0.158569,35942,290
0,0.05,0.080393,0.930076,0.147994,39407,259


# Threshold tuning for XGBoost

In [37]:
xgboost_inner = clone(xgb_model)
xgboost_inner.fit(X_inner_train_fe, y_inner_train)
inner_xgboost_scores = xgboost_inner.predict_proba(X_inner_val_fe)[:, 1]
xgb_threshold_results = []
for threshold in thresholds:
    y_pred_threshold = (inner_xgboost_scores >= threshold).astype(int)
    matrix = confusion_matrix(y_inner_val, y_pred_threshold)
    xgb_threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_inner_val, y_pred_threshold, zero_division=0),
        "recall": recall_score(y_inner_val, y_pred_threshold, zero_division=0),
        "f1": f1_score(y_inner_val, y_pred_threshold, zero_division=0),
        "false_positives": int(matrix[0, 1]),
        "false_negatives": int(matrix[1, 0]),
    })
xgb_threshold_results = pd.DataFrame(xgb_threshold_results).sort_values("f1", ascending=False)
xgb_threshold_results.to_csv(os.path.join(ARTIFACTS_PATH, "notebook_04_xgboost_inner_thresholds.csv"), index=False)
xgb_threshold_results.head(10)


,threshold,precision,recall,f1,false_positives,false_negatives
76,0.81,0.646552,0.506210,0.567838,1025,1829
78,0.83,0.681302,0.485961,0.567286,842,1904
77,0.82,0.663413,0.495410,0.567233,931,1869
79,0.84,0.696718,0.475702,0.565378,767,1942
75,0.80,0.627761,0.514039,0.565237,1129,1800
80,0.85,0.716132,0.463823,0.563002,681,1986
74,0.79,0.611129,0.521868,0.562982,1230,1771
81,0.86,0.736635,0.453834,0.561644,601,2023
73,0.78,0.595101,0.531317,0.561404,1339,1736
72,0.77,0.578016,0.538067,0.557327,1455,1711


In [38]:
best_xgb_threshold_row = xgb_threshold_results.iloc[0]
best_xgb_threshold_row

,76
threshold,0.810000
precision,0.646552
recall,0.506210
f1,0.567838
false_positives,1025.000000
false_negatives,1829.000000


In [39]:
xgb_high_recall_options = xgb_threshold_results[
  xgb_threshold_results["recall"] >= 0.85
].sort_values("precision", ascending=False)

xgb_high_recall_options.head(10)

,threshold,precision,recall,f1,false_positives,false_negatives
17,0.22,0.134525,0.855562,0.232493,20388,535
16,0.21,0.129790,0.860421,0.225556,21368,517
15,0.20,0.125596,0.866901,0.219406,22355,493
14,0.19,0.121535,0.873650,0.213386,23390,468
13,0.18,0.117715,0.881210,0.207686,24464,440
12,0.17,0.113537,0.887419,0.201317,25664,417
11,0.16,0.109499,0.892279,0.195060,26878,399
10,0.15,0.105455,0.897678,0.188738,28205,379
9,0.14,0.101524,0.902808,0.182523,29594,360
8,0.13,0.097761,0.908747,0.176531,31065,338


In [40]:
print("Best LightGBM threshold:")
display(best_threshold_row.to_frame().T)
print("\n")
print("Best XGBoost threshold:")
display(best_xgb_threshold_row.to_frame().T)

Best LightGBM threshold:


,threshold,precision,recall,f1,false_positives,false_negatives
74,0.79,0.622519,0.491091,0.549049,1103.0,1885.0




Best XGBoost threshold:


,threshold,precision,recall,f1,false_positives,false_negatives
76,0.81,0.646552,0.50621,0.567838,1025.0,1829.0


# Final model decision

The next cells select the model family by PR-AUC on this run's time holdout and report an operating threshold chosen on inner validation. Because the model family and earlier project choices have already been examined on the same final holdout, its score remains exploratory. A fresh later labeled period is needed for an untouched final estimate.

In [41]:
best_model_row = comparison_df.iloc[0]
print("Best model by PR-AUC:", best_model_row["model"])
print("Best PR-AUC:", best_model_row["pr_auc"])
print(comparison_df[["model", "roc_auc", "pr_auc", "precision", "recall", "f1"]])

Best model by PR-AUC: LightGBM
Best PR-AUC: 0.5358025508524761
                model   roc_auc    pr_auc  precision    recall        f1
0            LightGBM  0.904712  0.535803   0.321160  0.651329  0.430197
1             XGBoost  0.909574  0.534329   0.263953  0.705217  0.384131
2  LogisticRegression  0.844627  0.224230   0.105811  0.765748  0.185930


In [42]:
best_model_row = comparison_df.iloc[0]
best_model_name = best_model_row["model"]
if best_model_name == "LightGBM":
    chosen_threshold = best_threshold_row
    chosen_holdout_scores = y_val_proba_tree
elif best_model_name == "XGBoost":
    chosen_threshold = best_xgb_threshold_row
    chosen_holdout_scores = y_val_proba_xgb
else:
    chosen_threshold = None
    chosen_holdout_scores = y_val_proba

final_summary = {
    "best_model": best_model_name,
    "baseline_pr_auc": baseline_metrics["pr_auc"],
    "lightgbm_pr_auc": tree_metrics["pr_auc"],
    "xgboost_pr_auc": xgb_metrics["pr_auc"],
    "threshold_selection_period": "inner_training_validation",
}
if chosen_threshold is not None:
    threshold = float(chosen_threshold["threshold"])
    holdout_pred = (chosen_holdout_scores >= threshold).astype(int)
    final_summary.update({
        "best_threshold": threshold,
        "inner_threshold_precision": float(chosen_threshold["precision"]),
        "inner_threshold_recall": float(chosen_threshold["recall"]),
        "inner_threshold_f1": float(chosen_threshold["f1"]),
        "holdout_precision_at_selected_threshold": float(precision_score(y_val, holdout_pred, zero_division=0)),
        "holdout_recall_at_selected_threshold": float(recall_score(y_val, holdout_pred, zero_division=0)),
        "holdout_f1_at_selected_threshold": float(f1_score(y_val, holdout_pred, zero_division=0)),
    })
with open(os.path.join(ARTIFACTS_PATH, "notebook_04_final_summary.json"), "w") as f:
    json.dump(final_summary, f, indent=2)
final_summary


{'best_model': 'LightGBM',
 'baseline_pr_auc': np.float64(0.22423044790170896),
 'lightgbm_pr_auc': np.float64(0.5358025508524761),
 'xgboost_pr_auc': np.float64(0.5343289016923092),
 'threshold_selection_period': 'inner_training_validation',
 'best_threshold': 0.79,
 'inner_threshold_precision': 0.6225188227241615,
 'inner_threshold_recall': 0.49109071274298055,
 'inner_threshold_f1': 0.5490492001207365,
 'holdout_precision_at_selected_threshold': 0.6138580558358561,
 'holdout_recall_at_selected_threshold': 0.44906496062992124,
 'holdout_f1_at_selected_threshold': 0.5186869404575813}

In [43]:
print("Chosen final model:", best_model_name)

if best_model_name == "LightGBM":
    print("Chosen threshold row:")
    display(best_threshold_row.to_frame().T)
elif best_model_name == "XGBoost":
    print("Chosen threshold row:")
    display(best_xgb_threshold_row.to_frame().T)

Chosen final model: LightGBM
Chosen threshold row:


,threshold,precision,recall,f1,false_positives,false_negatives
74,0.79,0.622519,0.491091,0.549049,1103.0,1885.0


In [44]:
chosen_model_artifact = {
  "selected_model": best_model_name,
  "selection_metric": "pr_auc",
  "selected_threshold": final_summary.get("best_threshold"),
  "seed": SEED,
}

with open(os.path.join(ARTIFACTS_PATH, "notebook_04_chosen_model_artifact.json"), "w") as f:
  json.dump(chosen_model_artifact, f, indent=2)

chosen_model_artifact

{'selected_model': 'LightGBM',
 'selection_metric': 'pr_auc',
 'selected_threshold': 0.79,
 'seed': 42}

## Reproducibility notes

Rerun notebooks 2 and 3 before this one. They generate training-period-only feature lists and metadata. This notebook verifies the lists, uses a time-ordered 80/20 model split, and chooses thresholds on an inner time split within the earlier 80%. Model-family selection and final scores on the later 20% remain exploratory because that period was examined in earlier project runs.

# Store for future use

In [45]:
import joblib
import json
from pathlib import Path

MODEL_ARTIFACTS_PATH = Path(ARTIFACTS_PATH) / "models"
MODEL_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

if best_model_name == "LightGBM":
    final_model = tree_model
elif best_model_name == "XGBoost":
    final_model = xgb_model
elif best_model_name == "LogisticRegression":
    final_model = baseline_model
else:
    raise ValueError(f"Unknown best model: {best_model_name}")

model_path = MODEL_ARTIFACTS_PATH / "final_fraud_model.joblib"
metadata_path = MODEL_ARTIFACTS_PATH / "final_fraud_model_metadata.json"

joblib.dump(final_model, model_path)

model_metadata = {
    "best_model": best_model_name,
    "selection_metric": "pr_auc",
    "selected_threshold": final_summary.get("best_threshold"),
    "seed": SEED,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "selected_v_features": selected_v_features,
    "selected_id_features": selected_id_features,
    "v_selection_metadata": v_selection_metadata,
    "id_selection_metadata": id_selection_metadata,
    "target": TARGET,
    "drop_cols": DROP_COLS,
    "time_col": TIME_COL,
}

with open(metadata_path, "w") as f:
    json.dump(model_metadata, f, indent=2)

print("Saved model to:", model_path)
print("Saved metadata to:", metadata_path)

Saved model to: /content/drive/My Drive/ML/Fraud Detection/Data/Artifacts/models/final_fraud_model.joblib
Saved metadata to: /content/drive/My Drive/ML/Fraud Detection/Data/Artifacts/models/final_fraud_model_metadata.json
